# Feature Selection Cox (GitHub Ready Version)
This notebook has been cleaned of local absolute paths and prepared for GitHub distribution.
### Setup Instructions:
Before running this notebook, ensure your local directories are configured relative to this workspace or update the configuration paths below:
- Set up your data directory relative to the repository.
- Local path parameters have been updated to target `./data/`, `./Ensemble_model/`, or `./FHS60_pipeline/` by default.


### Coupling Note:
The second code cell (running the CoxNet cross-validation and feature selection) depends on in-memory variables generated by the first code cell (such as `X_train_scaled`, `y_train_struct`, `y_val_struct`, `event_train`, `cpg_cols`, etc.). 
Therefore, both cells must be executed sequentially from top to bottom in the same active session.

In [ ]:
import os
import datetime as dt
import logging
import gc

import numpy as np
import pandas as pd

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from joblib import dump
from sksurv.util import Surv

# ==========================================================
# 0. Paths & Directory Setup
# ==========================================================
DATA_PATH = r"./data/FHS_WHI_ComBat_batch_age_sex.parquet"
ROOT      = r"./Ensemble_model"

SAVED_MODELS_DIR = os.path.join(ROOT, "saved_models")
LOGS_DIR         = os.path.join(ROOT, "logs")
CSV_OUT_DIR      = os.path.join(ROOT, "outputs", "csv")
FIG_OUT_DIR      = os.path.join(ROOT, "outputs", "figures")

for d in [SAVED_MODELS_DIR, LOGS_DIR, CSV_OUT_DIR, FIG_OUT_DIR]:
    os.makedirs(d, exist_ok=True)

# ==========================================================
# 1. Logging Configuration
# ==========================================================
timestamp = dt.datetime.now().strftime("%Y%m%d_%H%M%S")
log_path = os.path.join(LOGS_DIR, f"data_prep_{timestamp}.log")

logging.basicConfig(
    filename=log_path,
    level=logging.INFO,
    format="%(asctime)s | %(levelname)s | %(message)s",
    datefmt="%Y-%m-%d %H:%M:%S",
)
logger = logging.getLogger(__name__)

def log_and_print(msg: str):
    print(msg)
    logger.info(msg)

log_and_print("===== Data preparation & split started =====")

# ==========================================================
# 2. Load Data
# ==========================================================
log_and_print(f"Loading data from: {DATA_PATH}")
df = pd.read_parquet(DATA_PATH)

log_and_print(f"Loaded DataFrame shape: {df.shape}")
log_and_print(f"Columns: {list(df.columns)[:10]} ... (total {len(df.columns)})")

# ==========================================================
# 3. Identify ID, Feature, Target, Metadata Columns
# ==========================================================
# ID column
if "Share_ID" in df.columns:
    id_col = "Share_ID"
elif "Patient_ID" in df.columns:
    id_col = "Patient_ID"
else:
    id_col = None
    log_and_print("[Warning] No Share_ID / Patient_ID column found. Using index as ID.")

# CpG features: columns starting with "cg"
cpg_cols = [c for c in df.columns if c.startswith("cg")]
log_and_print(f"Number of CpG feature columns: {len(cpg_cols)}")

# Targets
for col in ["time", "event"]:
    if col not in df.columns:
        raise ValueError(f"Required target column '{col}' not found in DataFrame.")

# Metadata columns
meta_cols = ["age", "sex", "batch"]
if id_col is not None:
    meta_cols = [id_col] + meta_cols

missing_meta = [c for c in meta_cols if c not in df.columns]
if missing_meta:
    raise ValueError(f"Missing metadata columns: {missing_meta}")

# Build full matrices
X_full    = df[cpg_cols].copy()
y_full    = df[["time", "event"]].copy()
meta_full = df[meta_cols].copy()

log_and_print(f"X_full shape (CpGs only): {X_full.shape}")
log_and_print(f"y_full shape: {y_full.shape}")
log_and_print(f"meta_full shape: {meta_full.shape}")

# Ensure event is integer / binary
y_full["event"] = y_full["event"].astype(int)

# ==========================================================
# 4. Train / Val / Test Split (60 / 20 / 20, stratified by event)
# ==========================================================
RANDOM_STATE = 42

event_all = y_full["event"].values

# Step 1: Train (60%) vs Temp (40%)
train_idx, temp_idx = train_test_split(
    np.arange(len(df)),
    test_size=0.4,
    stratify=event_all,
    random_state=RANDOM_STATE,
)

# Step 2: Val (20%) vs Test (20%) = 50/50 split of Temp
temp_events = event_all[temp_idx]
val_idx, test_idx = train_test_split(
    temp_idx,
    test_size=0.5,
    stratify=temp_events,
    random_state=RANDOM_STATE,
)

def describe_split(name, idx_array):
    n = len(idx_array)
    ev = y_full.iloc[idx_array]["event"].values
    n_events = ev.sum()
    n_cens = n - n_events
    log_and_print(
        f"{name}: N={n}, events={n_events}, censored={n_cens}, "
        f"event_rate={n_events / n:.3f}"
    )

describe_split("Train", train_idx)
describe_split("Validation", val_idx)
describe_split("Test", test_idx)

# Slice X, y, meta
X_train = X_full.iloc[train_idx].copy()
X_val   = X_full.iloc[val_idx].copy()
X_test  = X_full.iloc[test_idx].copy()

y_train = y_full.iloc[train_idx].copy()
y_val   = y_full.iloc[val_idx].copy()
y_test  = y_full.iloc[test_idx].copy()

meta_train = meta_full.iloc[train_idx].copy()
meta_val   = meta_full.iloc[val_idx].copy()
meta_test  = meta_full.iloc[test_idx].copy()

log_and_print(f"X_train shape: {X_train.shape}")
log_and_print(f"X_val shape  : {X_val.shape}")
log_and_print(f"X_test shape : {X_test.shape}")

# ==========================================================
# 5. StandardScaler Fit (Train only) and Transform
# ==========================================================
log_and_print("Fitting StandardScaler on TRAIN CpG features only.")
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_val_scaled   = scaler.transform(X_val)
X_test_scaled  = scaler.transform(X_test)

log_and_print("Scaling complete.")
log_and_print(f"X_train_scaled shape: {X_train_scaled.shape}")
log_and_print(f"X_val_scaled shape  : {X_val_scaled.shape}")
log_and_print(f"X_test_scaled shape : {X_test_scaled.shape}")

# ==========================================================
# 6. Save Scaler
# ==========================================================
scaler_path = os.path.join(SAVED_MODELS_DIR, "training_scaler.pkl")
dump(scaler, scaler_path)
log_and_print(f"Saved StandardScaler to: {scaler_path}")

# ==========================================================
# 7. Essential Survival Data Augmentations (no modeling yet)
# ==========================================================
log_and_print("Creating structured survival arrays (Surv) and event vectors...")

# Structured arrays for sksurv (required by Cox, RSF, etc.)
y_train_struct = Surv.from_dataframe("event", "time", y_train)
y_val_struct   = Surv.from_dataframe("event", "time", y_val)
y_test_struct  = Surv.from_dataframe("event", "time", y_test)

# Event vectors for stratified CV in future models
event_train = y_train["event"].astype(int).values
event_val   = y_val["event"].astype(int).values
event_test  = y_test["event"].astype(int).values

log_and_print("Survival augmentations complete.")

# (Optional) Save split indices for reproducibility/debugging
splits_csv = os.path.join(CSV_OUT_DIR, f"splits_{timestamp}.csv")
pd.DataFrame({
    "index": np.arange(len(df)),
    "split": np.where(
        np.isin(np.arange(len(df)), train_idx), "train",
        np.where(np.isin(np.arange(len(df)), val_idx), "val",
                 np.where(np.isin(np.arange(len(df)), test_idx), "test", "none"))
    )
}).to_csv(splits_csv, index=False)
log_and_print(f"Saved split assignment CSV: {splits_csv}")

log_and_print("===== Data preparation & split completed successfully =====")

# ==========================================================
# 8. Memory Cleanup (keep only what’s needed for modeling)
# ==========================================================
log_and_print("Cleaning up unscaled matrices to free memory...")

del df
del X_full
del y_full
del meta_full

del X_train
del X_val
del X_test

gc.collect()

log_and_print("Memory cleanup complete. Kept only scaled splits + metadata + Surv arrays.")

# At this point, the following variables are available for modeling:
#   cpg_cols                (list of CpG column names)
#   train_idx, val_idx, test_idx
#   X_train_scaled, X_val_scaled, X_test_scaled   (np.ndarray, CpGs only)
#   y_train, y_val, y_test                        (DataFrames with ['time', 'event'])
#   y_train_struct, y_val_struct, y_test_struct   (structured arrays for sksurv)
#   event_train, event_val, event_test            (event vectors for CV stratification)
#   meta_train, meta_val, meta_test               (ID / age / sex / batch)
#   scaler_path                                   (where the scaler is saved)


In [ ]:
import os
import time
import json
import datetime as dt
from collections import defaultdict

import numpy as np
import pandas as pd

from sklearn.model_selection import StratifiedKFold
from sksurv.metrics import concordance_index_censored
from sksurv.linear_model import CoxnetSurvivalAnalysis
import joblib

# ==========================================================
# 0. Paths (must match your project layout)
# ==========================================================
ROOT = r"./Ensemble_model"

LOG_DIR       = os.path.join(ROOT, "logs")
CSV_OUT_DIR   = os.path.join(ROOT, "outputs", "csv")
MODEL_OUT_DIR = os.path.join(ROOT, "saved_models")

os.makedirs(LOG_DIR, exist_ok=True)
os.makedirs(CSV_OUT_DIR, exist_ok=True)
os.makedirs(MODEL_OUT_DIR, exist_ok=True)

timestamp = dt.datetime.now().strftime("%Y%m%d_%H%M%S")
run_id    = f"coxnet_{timestamp}"
LOG_FILE  = os.path.join(LOG_DIR, f"{run_id}.log")

# ==========================================================
# 1. Logging helper
# ==========================================================
def log(msg: str):
    print(msg)
    with open(LOG_FILE, "a", encoding="utf-8") as f:
        f.write(f"{dt.datetime.now().isoformat()} | {msg}\n")

log("===== CoxNet (Elastic Net Cox) training started =====")

# ==========================================================
# 2. Check required variables from data prep
# ==========================================================
required_vars = [
    "X_train_scaled", "X_val_scaled", "X_test_scaled",
    "y_train_struct", "y_val_struct", "y_test_struct",
    "y_train", "y_val", "y_test",
    "event_train", "cpg_cols"
]

missing = [v for v in required_vars if v not in globals()]
if missing:
    raise RuntimeError(f"Missing required variables from data prep: {missing}")

X_train_s = X_train_scaled
X_val_s   = X_val_scaled
X_test_s  = X_test_scaled

log(f"X_train_scaled shape: {X_train_s.shape}")
log(f"X_val_scaled shape  : {X_val_s.shape}")
log(f"X_test_scaled shape : {X_test_s.shape}")
log(f"Number of CpG features: {len(cpg_cols)}")

# ==========================================================
# 3. Metric: Concordance Index (using structured arrays)
# ==========================================================
def cindex_sksurv(y_struct, scores):
    """
    y_struct: structured array with fields ('event', 'time')
    scores  : risk scores (higher = higher risk)
    """
    return float(
        concordance_index_censored(
            y_struct["event"],
            y_struct["time"],
            scores
        )[0]
    )

# ==========================================================
# 4. Coefficient extractor (robust to coef_path_ vs coef_)
# ==========================================================
def get_coefficients_for_alpha(model, alpha, feature_names: pd.Index):
    alphas_attr = getattr(model, "alphas_", None)
    coef_path   = getattr(model, "coef_path_", None)
    coef_single = getattr(model, "coef_", None)

    if coef_path is not None and alphas_attr is not None and coef_path.ndim == 2:
        j = int(np.argmin(np.abs(np.asarray(alphas_attr, dtype=float) - float(alpha))))
        coef_vec = coef_path[:, j]
    elif coef_single is not None:
        coef_vec = np.asarray(coef_single, dtype=float).ravel()
    else:
        raise RuntimeError("CoxNet: coefficients not available (no coef_path_ or coef_).")

    return pd.DataFrame({"feature": feature_names, "coefficient": coef_vec})

# ==========================================================
# 5. Hyperparameter grid and CV settings
# ==========================================================
L1_GRID         = [0.3]   # you can adjust this
ALPHA_MIN_RATIO = 0.35
N_ALPHAS_PATH   = 100

N_SPLITS     = 5
SHUFFLE      = True
RANDOM_STATE = 42

MAX_ITER = 1000
TOL      = 1e-6

# ==========================================================
# 6. Build alpha path from TRAIN only
# ==========================================================
log("Building alpha path from TRAIN data...")

alpha_builder = CoxnetSurvivalAnalysis(
    l1_ratio=0.5,                    # only for path-building
    n_alphas=N_ALPHAS_PATH,
    alpha_min_ratio=ALPHA_MIN_RATIO,
    max_iter=MAX_ITER,
    tol=TOL,
    fit_baseline_model=False,
)

alpha_builder.fit(X_train_s, y_train_struct)
alpha_grid = alpha_builder.alphas_.copy()

# Optional thinning to speed up CV
if alpha_grid.shape[0] > 60:
    idx = np.unique(np.linspace(0, len(alpha_grid) - 1, 60).round().astype(int))
    alpha_grid = alpha_grid[idx]

log(f"Alpha grid size after thinning: {len(alpha_grid)} "
    f"(min={alpha_grid.min():.3e}, max={alpha_grid.max():.3e})")

# ==========================================================
# 7. K-fold CV over (l1_ratio, alpha)
# ==========================================================
log("Starting K-fold CV...")

skf = StratifiedKFold(
    n_splits=N_SPLITS,
    shuffle=SHUFFLE,
    random_state=RANDOM_STATE,
)

param_scores = defaultdict(list)
cv_rows      = []
fold_rows    = []

n_total   = len(L1_GRID) * len(alpha_grid) * N_SPLITS
counter   = 0
t0_global = time.time()

for l1 in L1_GRID:
    log(f"--- Evaluating l1_ratio={l1} ---")
    running_mean = 0.0
    n_seen = 0

    for fold_num, (tr_idx, va_idx) in enumerate(skf.split(X_train_s, event_train), start=1):
        X_tr, X_va = X_train_s[tr_idx], X_train_s[va_idx]
        y_tr, y_va = y_train_struct[tr_idx], y_train_struct[va_idx]

        est = CoxnetSurvivalAnalysis(
            l1_ratio=l1,
            alphas=alpha_grid,
            max_iter=MAX_ITER,
            tol=TOL,
            fit_baseline_model=False,
        )
        est.fit(X_tr, y_tr)

        for alpha in est.alphas_:
            counter += 1
            preds = est.predict(X_va, alpha=float(alpha))
            cidx  = cindex_sksurv(y_va, preds)

            param_scores[(l1, float(alpha))].append(cidx)

            running_mean = (running_mean * n_seen + cidx) / (n_seen + 1)
            n_seen += 1

            fold_rows.append({
                "run_id": run_id,
                "fold": fold_num,
                "l1_ratio": l1,
                "alpha": float(alpha),
                "val_cindex": cidx,
            })

            if counter % 50 == 0:
                elapsed_min = (time.time() - t0_global) / 60.0
                log(f"[{counter}/{n_total}] l1={l1}, fold={fold_num}, "
                    f"alpha={alpha:.3e}, cidx={cidx:.4f}, "
                    f"running_mean_l1={running_mean:.4f}, "
                    f"time={elapsed_min:.1f} min")

# ==========================================================
# 8. Aggregate CV results and select best (l1, alpha)
# ==========================================================
log("Aggregating CV results...")

for (l1, alpha), scores in param_scores.items():
    cv_rows.append({
        "run_id": run_id,
        "l1_ratio": l1,
        "alpha": alpha,
        "n_folds": len(scores),
        "mean_val_cindex": float(np.mean(scores)),
        "std_val_cindex": float(np.std(scores, ddof=1)) if len(scores) > 1 else 0.0,
        "min_val_cindex": float(np.min(scores)),
        "max_val_cindex": float(np.max(scores)),
    })

cv_df   = pd.DataFrame(cv_rows).sort_values(
    ["mean_val_cindex", "l1_ratio", "alpha"],
    ascending=[False, True, True]
)
fold_df = pd.DataFrame(fold_rows)

CV_RESULTS_CSV   = os.path.join(CSV_OUT_DIR, f"{run_id}_cv_summary.csv")
FOLDS_RESULTS_CSV = os.path.join(CSV_OUT_DIR, f"{run_id}_cv_folds.csv")

cv_df.to_csv(CV_RESULTS_CSV, index=False)
fold_df.to_csv(FOLDS_RESULTS_CSV, index=False)

best_row   = cv_df.iloc[0]
best_l1    = float(best_row["l1_ratio"])
best_alpha = float(best_row["alpha"])
best_mean  = float(best_row["mean_val_cindex"])

log(f"BEST params → l1_ratio={best_l1}, alpha={best_alpha:.3e}, "
    f"mean CV C-index={best_mean:.4f}")

# ==========================================================
# 9. Refit final model on FULL TRAIN
# ==========================================================
log("Refitting final CoxNet model on FULL TRAIN...")

final_est = CoxnetSurvivalAnalysis(
    l1_ratio=best_l1,
    alphas=[best_alpha],
    max_iter=MAX_ITER,
    tol=TOL,
    fit_baseline_model=False,
)
final_est.fit(X_train_s, y_train_struct)

# ==========================================================
# 10. Evaluate on Validation and Test
# ==========================================================
log("Evaluating final model on VAL and TEST splits...")

val_pred  = final_est.predict(X_val_s,  alpha=best_alpha)
test_pred = final_est.predict(X_test_s, alpha=best_alpha)

val_c  = cindex_sksurv(y_val_struct,  val_pred)
test_c = cindex_sksurv(y_test_struct, test_pred)

log(f"Validation C-index (held-out): {val_c:.4f}")
log(f"Test       C-index (held-out): {test_c:.4f}")

# ==========================================================
# 11. Save coefficient tables
# ==========================================================
log("Extracting and saving coefficient tables...")

feature_names = pd.Index(cpg_cols)  # aligned with columns of X_train_scaled
coef_df = get_coefficients_for_alpha(final_est, best_alpha, feature_names)
coef_df = coef_df.sort_values("coefficient", key=lambda s: np.abs(s), ascending=False)

COEF_FULL_CSV = os.path.join(CSV_OUT_DIR, f"{run_id}_coefficients_full.csv")
COEF_NZ_CSV   = os.path.join(CSV_OUT_DIR, f"{run_id}_coefficients_nonzero.csv")

coef_df.to_csv(COEF_FULL_CSV, index=False)
nonzero_df = coef_df[coef_df["coefficient"] != 0.0].copy()
nonzero_df.to_csv(COEF_NZ_CSV, index=False)

log(f"Saved ALL coefficients:     {COEF_FULL_CSV} (n={coef_df.shape[0]})")
log(f"Saved NON-ZERO coefficients: {COEF_NZ_CSV} (n={nonzero_df.shape[0]})")

# ==========================================================
# 12. Save final model + metadata
# ==========================================================
MODEL_PATH = os.path.join(MODEL_OUT_DIR, f"{run_id}.pkl")

metadata = {
    "run_id": run_id,
    "timestamp": dt.datetime.now().isoformat(timespec="seconds"),
    "model": "CoxnetSurvivalAnalysis",
    "best_l1_ratio": best_l1,
    "best_alpha": best_alpha,
    "alpha_min_ratio_for_path": ALPHA_MIN_RATIO,
    "n_alphas_path": N_ALPHAS_PATH,
    "cv_n_splits": N_SPLITS,
    "cv_mean_val_cindex": best_mean,
    "heldout_val_cindex": val_c,
    "heldout_test_cindex": test_c,
    "n_features_total": int(coef_df.shape[0]),
    "n_features_nonzero": int(nonzero_df.shape[0]),
    "paths": {
        "cv_summary_csv": CV_RESULTS_CSV,
        "cv_folds_csv": FOLDS_RESULTS_CSV,
        "coef_full_csv": COEF_FULL_CSV,
        "coef_nonzero_csv": COEF_NZ_CSV,
        "log_file": LOG_FILE,
    },
}

joblib.dump(
    {"estimator": final_est, "chosen_alpha": best_alpha, "metadata": metadata},
    MODEL_PATH
)

# Also save a small JSON metadata copy
META_JSON = os.path.join(CSV_OUT_DIR, f"{run_id}_metadata.json")
with open(META_JSON, "w", encoding="utf-8") as f:
    json.dump(metadata, f, indent=2)

log(f"Saved final model: {MODEL_PATH}")
log(f"Saved metadata JSON: {META_JSON}")
log("===== CoxNet training completed successfully =====")
